# 06 · When retrieval fails

## Goal

Measure recall and precision on the harness instead of eyeballing answers,
tune knowledge-source `description` fields as the selection lever once you
have more than a couple of sources, and turn the ungrounded-response policy
from a checkbox into a tradeoff you've actually seen the cost of both ways.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)


## Concept

Recall failures (the answer exists somewhere in a source but didn't surface)
and precision failures (an answer surfaced but from the wrong source, or
partially wrong) look identical from the chat window — both are "the agent
got it wrong." They need different fixes: recall failures usually mean
chunking or source description tuning; precision failures usually mean
overlapping source descriptions confusing selection, which only shows up
once you're past 2-3 sources (Meridian's MSA, the addenda index, and the
SharePoint library all plausibly answer "what's Meridian's payment term").

**Ungrounded-response policy** — whether the agent may answer from general
knowledge when no source has the fact — is a measured tradeoff: allow it
and precision on `know-05-ungrounded-policy`-style cases drops; forbid it
and recall on edge-of-source questions drops, because the agent declines
things it could plausibly reason about. This notebook doesn't tell you
which to pick; it makes you look at the number before you pick.


## Build


### Tighten source descriptions for selection


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

# Before: descriptions were generic ("contract info"). After: scoped enough
# that the selector doesn't have to guess between three sources that could
# all plausibly answer a payment-terms question.
for fname, new_description in [
    ("meridian-msa.yaml", "The signed master services agreement for Meridian Cables specifically — notice periods, base pricing, termination. Not addenda, not other suppliers."),
    ("addenda-search-index.yaml", "Scanned pricing/terms addenda that amend a supplier's base contract — check here for updates, not the original MSA."),
    ("supplier-contracts-sharepoint.yaml", "The authoritative, access-controlled library for all suppliers' contracts and confidential pricing annexes — check here when a supplier isn't Meridian Cables."),
]:
    path = workspace / "knowledge" / fname
    doc = yaml.safe_load(path.read_text())
    doc["description"] = new_description
    path.write_text(yaml.dump(doc, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### Set (and record) the ungrounded-response policy


In [ ]:
policy_doc = workspace / "copilot.yaml"
text = policy_doc.read_text()
if "ungroundedResponsePolicy" not in text:
    text += "\nungroundedResponsePolicy: decline\n"  # 'decline' | 'allow-with-disclaimer'
    policy_doc.write_text(text)
print(text)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

recall_cases = load_golden(tags=["recall"])
precision_cases = load_golden(tags=["precision"])
ungrounded_cases = load_golden(tags=["ungrounded"])

recall_suite = run_suite(client, cases=recall_cases, credit_meter=meter, min_pass_rate=0.8)
precision_suite = run_suite(client, cases=precision_cases, credit_meter=meter, min_pass_rate=0.8)
ungrounded_suite = run_suite(client, cases=ungrounded_cases, credit_meter=meter, min_pass_rate=0.8)

core_suite = run_suite(client, cases=load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


Compare this run's recall/precision numbers against `03`-`05`'s runs (same cases, different description tuning) — that delta is the actual measurement this notebook produces.


## Cost


In [ ]:
total = recall_suite.total_credits + precision_suite.total_credits + ungrounded_suite.total_credits + core_suite.total_credits
meter.report_cost("06", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=total, note="description tuning + recall/precision/ungrounded measurement")


## Teardown


In [ ]:
print("No teardown — description tuning and policy setting persist.")
